# 02_feature_snapshots

**Owner:** Data and replay. **Blueprint step:** 2. **Status:** Runs locally. Not yet run in Fabric.

Builds frozen feature snapshots. A checkpoint snapshot is taken 90 days before each deal end. A live snapshot is taken at a chosen cutoff and uses staged journey results from `pl_export_journeys`.

- INV-2: every input has its event time and its ingestion time at or before `cutoff_at`.
- A snapshot is immutable per `snapshot_id`. Rewriting one with different content fails.
- Observed outcomes are written to a separate table. They never enter a snapshot.

Output: `lakehouse/features/feature_snapshot.parquet` and `lakehouse/features/observed_outcome.parquet`. All data is synthetic.

In [ ]:
OUTPUT_ROOT = None  # default: data/output
MODE = 'checkpoint'  # 'checkpoint' or 'live'
CUTOFF = None  # live mode: cutoff time (ISO 8601)
MORTGAGE_IDS = None  # live mode: optional list of mortgage_ids

In [ ]:
import sys
from pathlib import Path

# Local run: this notebook sits in notebooks/ and the package in data/.
# Fabric run (not yet done): attach the Lakehouse and set DATA_DIR and OUTPUT_ROOT to its Files paths.
DATA_DIR = next(p for p in [Path.cwd() / "data", Path.cwd().parent / "data"] if (p / "hbdata").exists())
sys.path.insert(0, str(DATA_DIR))
OUTPUT_ROOT = Path(OUTPUT_ROOT) if OUTPUT_ROOT else DATA_DIR / "output"
LAKEHOUSE = OUTPUT_ROOT / "lakehouse"
import json
import pandas as pd
manifest = json.loads((OUTPUT_ROOT / "exports" / "manifest.json").read_text(encoding="utf-8"))
AS_OF = pd.Timestamp(manifest["config"]["as_of"])
print("Output root:", OUTPUT_ROOT, "| generator as_of:", AS_OF)

In [ ]:
from hbdata import io, pipelines_local, snapshots

clean = pipelines_local.read_clean(LAKEHOUSE)
if MODE == 'live':
    cutoff = pd.Timestamp(CUTOFF)
    requests = snapshots.live_requests(clean['mortgage_account'], cutoff, MORTGAGE_IDS)
    staged = pipelines_local.read_staged_events(LAKEHOUSE)
    created_at = cutoff
else:
    requests = snapshots.checkpoint_requests(clean['mortgage_account'], AS_OF)
    staged = None
    created_at = AS_OF
snap = snapshots.build_snapshots(clean, requests, created_at, staged)
stats = snapshots.append_snapshots(snap, LAKEHOUSE / 'features' / 'feature_snapshot.parquet')
io.write_table(clean['observed_outcome'], LAKEHOUSE / 'features' / 'observed_outcome.parquet', 'observed_outcome')
stats

## INV-2 checks

Every source watermark in a snapshot must be at or before its cutoff.

In [ ]:
wm = snap['source_watermark'].map(json.loads)
for name in snapshots.SOURCES:
    t = pd.to_datetime(wm.map(lambda d: d[name]), utc=True, format="ISO8601")
    assert (t.isna() | (t <= snap['cutoff_at'])).all(), name
assert not [c for c in snap.columns if 'outcome' in c or c == 'label']
print('INV-2 watermark checks passed for', len(snap), 'snapshots')

## Summary by deal-end cohort

In [ ]:
snap.groupby('cohort_id').agg(snapshots=('snapshot_id', 'count'), started=('pt_request_started', 'mean'),
    software_faults=('pt_software_errors_90d', 'mean'), unresolved_complaint=('svc_unresolved_complaint', 'mean'),
    payment_feed_stale=('ctx_payment_feed_stale', 'sum')).round(3)

## Illustrative personas

Alex, Sam, Casey and Robin are fictional demo personas (see `data/README.md`).

In [ ]:
cols = ['mortgage_id', 'snapshot_kind', 'cutoff_at', 'pt_latest_stage', 'pt_software_errors_90d', 'pt_unrecovered_software_error',
        'svc_contacts_90d', 'svc_max_contacts_per_case_365d', 'svc_repeat_contacts_unresolved', 'svc_unresolved_complaint', 'ctx_reversals_180d']
snap[snap['mortgage_id'].str.startswith('HB-M-9000')][cols]